In [ ]:

import numpy as np
import pandas as pd
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 60)
print("FetalAI - Model Training")
print("=" * 60)

DATA_PATH = os.path.join(os.path.dirname(__file__), '..', 'fetal_health.csv')

if os.path.exists(DATA_PATH):
    data = pd.read_csv(DATA_PATH)
    print(f"✓ Loaded dataset: {DATA_PATH}")
else:
    print("⚠  fetal_health.csv not found – generating synthetic demo data.")
    print("   Download the real dataset from:")
    print("   https://www.kaggle.com/datasets/andrewmvd/fetal-health-classification")
    print()

    rng = np.random.default_rng(42)
    n1, n2, n3 = 1655, 295, 176

    def make_class(rng, n, label, acc_m, astv_m, pct_m, hvar_m, hmed_m, mltv_m, hmode_m, pdec_m):
        return pd.DataFrame({
            'baseline value':           rng.normal(133, 9.8, n).clip(106, 160),
            'accelerations':            rng.normal(acc_m, acc_m*0.5+0.0005, n).clip(0, 0.019),
            'fetal_movement':           rng.exponential(0.009, n).clip(0, 0.481),
            'uterine_contractions':     rng.exponential(0.004, n).clip(0, 0.015),
            'light_decelerations':      rng.exponential(0.002, n).clip(0, 0.015),
            'severe_decelerations':     rng.exponential(0.000003, n).clip(0, 0.001),
            'prolongued_decelerations': rng.exponential(max(pdec_m,1e-7), n).clip(0, 0.005),
            'abnormal_short_term_variability': rng.normal(astv_m, 10, n).clip(12, 87),
            'mean_value_of_short_term_variability': rng.normal(1.33, 0.88, n).clip(0.2, 7.0),
            'percentage_of_time_with_abnormal_long_term_variability':
                                        rng.exponential(max(pct_m,0.5), n).clip(0, 91),
            'mean_value_of_long_term_variability': rng.normal(mltv_m, 4, n).clip(0, 50),
            'histogram_width':          rng.normal(70, 35, n).clip(3, 180),
            'histogram_min':            rng.normal(93, 28, n).clip(50, 159),
            'histogram_max':            rng.normal(164, 18, n).clip(122, 238),
            'histogram_number_of_peaks': rng.poisson(4, n).clip(0, 18).astype(float),
            'histogram_number_of_zeroes': rng.poisson(0.32, n).clip(0, 10).astype(float),
            'histogram_mode':           rng.normal(hmode_m, 14, n).clip(60, 187),
            'histogram_median':         rng.normal(hmed_m, 12, n).clip(77, 186),
            'histogram_variance':       rng.exponential(max(hvar_m,1), n).clip(0, 269),
            'histogram_tendency':       rng.choice([-1., 0., 1.], n, p=[0.09, 0.8, 0.11]),
            'fetal_health':             np.full(n, float(label)),
        })

    # acc, astv, pct_altv, hvar, hmed, mltv, hmode, pdec
    c1 = make_class(rng, n1, 1, 0.004, 38,  3,  10, 141,  9.5, 141, 0.00002)
    c2 = make_class(rng, n2, 2, 0.002, 57, 18,  32, 144,  6.8, 137, 0.0004)
    c3 = make_class(rng, n3, 3, 0.001, 72, 45,  65, 147,  4.5, 132, 0.0012)

    data = pd.concat([c1, c2, c3], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"✓ Synthetic dataset created: {len(data)} rows, {data.shape[1]} columns")

print(f"\nDataset shape : {data.shape}")
print(f"\nTarget distribution:\n{data['fetal_health'].value_counts().sort_index()}")
print("\n── Descriptive Statistics ──")
print(data.describe().T[['mean','std','min','max']].round(3))

os.makedirs('plots', exist_ok=True)

plt.figure(figsize=(18, 14))
sns.heatmap(data.corr(numeric_only=True), annot=False, cmap='Greens', center=0, linewidths=0.3)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig('plots/correlation_matrix.png', dpi=100)
plt.close()

plt.figure(figsize=(5, 4))
colours = ["#f7b2b0", "#8f7198", "#003f5c"]
sns.countplot(data=data, x='fetal_health', palette=colours)
plt.title('Class Distribution')
plt.tight_layout()
plt.savefig('plots/class_distribution.png', dpi=100)
plt.close()
print("✓ Plots saved to ./plots/")

if 'histogram_mean' in data.columns:
    data.drop(columns=['histogram_mean'], inplace=True)

FEATURES = [
    'prolongued_decelerations',
    'abnormal_short_term_variability',
    'percentage_of_time_with_abnormal_long_term_variability',
    'histogram_variance',
    'histogram_median',
    'mean_value_of_long_term_variability',
    'histogram_mode',
    'accelerations',
]
FEATURES = [f for f in FEATURES if f in data.columns]
print(f"\n✓ Selected {len(FEATURES)} features: {FEATURES}")

X = data[FEATURES]
y = data['fetal_health']

scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=10, stratify=y)
print(f"\nTrain size : {X_train.shape}  |  Test size : {X_test.shape}")

print("\n── Training Models ──")
models = {
    'Random Forest Classifier': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'Decision Tree Classifier': DecisionTreeClassifier(class_weight='balanced', random_state=42),
    'Logistic Regression':      LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'K Neighbors Classifier':   KNeighborsClassifier(n_neighbors=5),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results[name] = {'model': model, 'accuracy': acc, 'predictions': preds}
    print(f"  {name:35s}  Accuracy: {acc:.4f}")

best_name = max(results, key=lambda k: results[k]['accuracy'])
best_model = results[best_name]['model']
best_preds = results[best_name]['predictions']

print(f"\n✓ Best model: {best_name}  (Accuracy: {results[best_name]['accuracy']:.4f})")
print("\nClassification Report:")
print(classification_report(y_test, best_preds,
                             target_names=['Normal','Suspect','Pathological']))

cm = confusion_matrix(y_test, best_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Normal','Suspect','Pathological'])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=True)
plt.title(f'Confusion Matrix – {best_name}')
plt.tight_layout()
plt.savefig('plots/confusion_matrix.png', dpi=100)
plt.close()

comparison_df = pd.DataFrame(
    [{'name': k, 'score': v['accuracy']} for k, v in results.items()]
).sort_values('score', ascending=True)
plt.figure(figsize=(8, 4))
sns.barplot(data=comparison_df, y='name', x='score', palette='viridis')
plt.xlim(0, 1)
plt.title('Model Accuracy Comparison')
plt.xlabel('Accuracy')
plt.tight_layout()
plt.savefig('plots/model_comparison.png', dpi=100)
plt.close()
print("✓ Confusion matrix and comparison charts saved to ./plots/")

MODEL_DIR = os.path.join(os.path.dirname(__file__), '..', 'flask')
os.makedirs(MODEL_DIR, exist_ok=True)

with open(os.path.join(MODEL_DIR, 'fetal_health1.pkl'), 'wb') as f:
    pickle.dump(best_model, f)
with open(os.path.join(MODEL_DIR, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
with open(os.path.join(MODEL_DIR, 'features.pkl'), 'wb') as f:
    pickle.dump(FEATURES, f)

print(f"\n✓ Model saved  → flask/fetal_health1.pkl")
print(f"✓ Scaler saved → flask/scaler.pkl")
print(f"✓ Features saved → flask/features.pkl")
print("\n" + "=" * 60)
print("Training complete! Run the Flask app with: python flask/app.py")
print("=" * 60)
